[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/08-pose-estimation.ipynb)

# Module 6.8 — Pose Estimation and MediaPipe
**Module 6: Computer Vision** | Estimated time: 25 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Install and configure MediaPipe solutions in Colab
- Detect and visualise full-body pose landmarks with `mp.solutions.pose`
- Track hand landmarks and count extended fingers
- Apply `mp.solutions.face_mesh` to extract 468 facial landmarks
- Build gesture recognition logic (thumbs up / thumbs down detection)

In [ ]:
!pip install mediapipe --quiet

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import requests, os

print(f'MediaPipe version: {mp.__version__}')
print(f'OpenCV version  : {cv2.__version__}')

os.makedirs('/tmp/cv_pose', exist_ok=True)

def show(img_bgr, title='', figsize=(8, 6)):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb); plt.title(title, fontsize=11)
    plt.axis('off'); plt.tight_layout(); plt.show()

# Download sample images
urls = {
    'person.jpg':   'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/220px-Gatto_europeo4.jpg',
    'hand.jpg':     'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/220px-Gatto_europeo4.jpg',
    'face.jpg':     'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/220px-Gatto_europeo4.jpg',
}
# Use a real person photo if available; fall back to synthesis
person_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/9/9b/Gustav_Klimt_046.jpg/220px-Gustav_Klimt_046.jpg'
for fname, url in [('person.jpg', person_url), ('hand.jpg', person_url), ('face.jpg', person_url)]:
    r = requests.get(url)
    with open(f'/tmp/cv_pose/{fname}', 'wb') as f:
        f.write(r.content)
print('Sample images downloaded.')

## MediaPipe Pose: Full-Body Landmark Detection

MediaPipe Pose detects 33 landmarks on the human body. Each landmark has (x, y, z, visibility) coordinates normalised to [0, 1].

**Key landmark indices:**
```
0  — Nose          11, 12 — Shoulders
13, 14 — Elbows    15, 16 — Wrists
23, 24 — Hips      25, 26 — Knees
27, 28 — Ankles
```

MediaPipe processes RGB images. Always convert from BGR before calling `.process()`.

In [ ]:
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

# Landmark name mapping (subset)
LANDMARK_NAMES = {
    0:  'NOSE',
    11: 'LEFT_SHOULDER',  12: 'RIGHT_SHOULDER',
    13: 'LEFT_ELBOW',     14: 'RIGHT_ELBOW',
    15: 'LEFT_WRIST',     16: 'RIGHT_WRIST',
    23: 'LEFT_HIP',       24: 'RIGHT_HIP',
    25: 'LEFT_KNEE',      26: 'RIGHT_KNEE',
    27: 'LEFT_ANKLE',     28: 'RIGHT_ANKLE',
}

def run_pose(image_path, min_detection_confidence=0.5):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        # Synthetic standing-figure image
        img_bgr = np.ones((480, 320, 3), dtype=np.uint8) * 200
        # Head
        cv2.circle(img_bgr, (160, 60), 35, (200, 170, 140), -1)
        # Body
        cv2.line(img_bgr, (160, 95), (160, 260), (150, 100, 80), 5)
        # Arms
        cv2.line(img_bgr, (160, 130), (80, 200), (150, 100, 80), 4)
        cv2.line(img_bgr, (160, 130), (240, 200), (150, 100, 80), 4)
        # Legs
        cv2.line(img_bgr, (160, 260), (120, 380), (150, 100, 80), 5)
        cv2.line(img_bgr, (160, 260), (200, 380), (150, 100, 80), 5)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=1,
        min_detection_confidence=min_detection_confidence
    ) as pose:
        results = pose.process(img_rgb)

    annotated = img_bgr.copy()
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            annotated,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style()
        )
        # Print key landmarks
        h, w = img_bgr.shape[:2]
        print('Key landmark positions (normalised):')
        for idx, name in list(LANDMARK_NAMES.items())[:6]:
            lm = results.pose_landmarks.landmark[idx]
            print(f'  {name:20s}  x={lm.x:.3f}  y={lm.y:.3f}  '
                  f'visibility={lm.visibility:.2f}')
    else:
        print('No pose detected. Using synthetic pose for illustration.')
        # Draw synthetic skeleton for demonstration
        pts = {0:(160,60), 11:(100,130), 12:(220,130),
               13:(70,190), 14:(250,190), 15:(50,250), 16:(270,250),
               23:(120,270), 24:(200,270), 25:(110,360), 26:(210,360),
               27:(105,440), 28:(215,440)}
        for (a, b) in [(0,11),(0,12),(11,12),(11,13),(13,15),
                       (12,14),(14,16),(11,23),(12,24),(23,24),
                       (23,25),(25,27),(24,26),(26,28)]:
            if a in pts and b in pts:
                cv2.line(annotated, pts[a], pts[b], (0,255,0), 3)
        for idx, pt in pts.items():
            cv2.circle(annotated, pt, 6, (0,0,255), -1)

    return annotated, results

annotated, pose_results = run_pose('/tmp/cv_pose/person.jpg')
show(annotated, 'MediaPipe Pose — 33 Body Landmarks')

## MediaPipe Hands: Hand Landmark Detection

MediaPipe Hands detects up to two hands and returns 21 landmarks per hand. The landmarks follow the hand skeleton from wrist (0) through each finger tip (4, 8, 12, 16, 20).

We can count extended fingers by comparing the y-coordinate of each fingertip to its lower joint — if the tip is above the pip joint, the finger is extended.

In [ ]:
mp_hands = mp.solutions.hands

# Finger tip and pip (proximal interphalangeal) landmark indices
FINGER_TIPS = [4, 8, 12, 16, 20]   # Thumb, Index, Middle, Ring, Pinky
FINGER_PIPS = [3, 6, 10, 14, 18]   # One joint below each tip

def count_fingers(hand_landmarks, handedness='Right'):
    """Count extended fingers. Returns count and list of booleans."""
    lm = hand_landmarks.landmark
    extended = []

    # Thumb: compare x coordinates (horizontal)
    if handedness == 'Right':
        thumb_open = lm[4].x < lm[3].x   # tip to the left of pip
    else:
        thumb_open = lm[4].x > lm[3].x
    extended.append(thumb_open)

    # Other fingers: compare y (tip above pip = finger up)
    for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:]):
        extended.append(lm[tip].y < lm[pip].y)

    return sum(extended), extended

def run_hands(image_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        img_bgr = np.ones((400, 400, 3), dtype=np.uint8) * 230
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_bgr.shape[:2]

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=2,
        min_detection_confidence=0.5
    ) as hands:
        results = hands.process(img_rgb)

    annotated = img_bgr.copy()
    if results.multi_hand_landmarks:
        for hand_lm, hand_info in zip(results.multi_hand_landmarks,
                                       results.multi_handedness):
            mp_drawing.draw_landmarks(
                annotated, hand_lm, mp_hands.HAND_CONNECTIONS)
            side  = hand_info.classification[0].label
            count, ext = count_fingers(hand_lm, side)
            # Wrist position for label
            wx = int(hand_lm.landmark[0].x * w)
            wy = int(hand_lm.landmark[0].y * h)
            cv2.putText(annotated, f'{side}: {count} fingers',
                        (wx - 60, wy - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            finger_names = ['Thumb','Index','Middle','Ring','Pinky']
            print(f'{side} hand ({count} extended):', end=' ')
            print(', '.join([n for n, e in zip(finger_names, ext) if e]))
    else:
        print('No hands detected in this image.')
        cv2.putText(annotated, 'No hands detected',
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    return annotated, results

hand_annotated, hand_results = run_hands('/tmp/cv_pose/hand.jpg')
show(hand_annotated, 'MediaPipe Hands — 21 Landmarks per Hand')

## MediaPipe Face Mesh: 468 Facial Landmarks

Face Mesh builds a dense 3D mesh of the face with 468 landmarks. These are used for face filters, emotion recognition, gaze estimation, and facial action unit analysis.

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

# Named landmark indices (subset)
FACE_LANDMARKS = {
    'Nose tip':       1,
    'Left eye outer': 33,
    'Right eye outer':263,
    'Left mouth':     61,
    'Right mouth':    291,
    'Chin':           152,
    'Left eyebrow':   70,
    'Right eyebrow':  300,
}

def run_face_mesh(image_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        img_bgr = np.ones((400, 320, 3), dtype=np.uint8) * 210
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_bgr.shape[:2]

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:
        results = face_mesh.process(img_rgb)

    annotated = img_bgr.copy()
    if results.multi_face_landmarks:
        for face_lm in results.multi_face_landmarks:
            # Draw all mesh connections (tessellation)
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=face_lm,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_styles.get_default_face_mesh_tesselation_style()
            )
            # Draw contours
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=face_lm,
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_styles.get_default_face_mesh_contours_style()
            )
            # Annotate key points
            for name, idx in FACE_LANDMARKS.items():
                lm = face_lm.landmark[idx]
                px, py = int(lm.x * w), int(lm.y * h)
                cv2.circle(annotated, (px, py), 4, (0, 255, 255), -1)
                cv2.putText(annotated, name, (px + 5, py - 3),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 0), 1)
        print(f'Face detected — {len(results.multi_face_landmarks[0].landmark)} landmarks')
    else:
        print('No face detected in this image.')
    return annotated, results

face_annotated, face_results = run_face_mesh('/tmp/cv_pose/face.jpg')
show(face_annotated, 'MediaPipe Face Mesh — 468 Landmarks')

## Gesture Recognition: Thumbs Up / Thumbs Down

Using hand landmarks we can build rule-based gesture detectors. The key insight:
- **Thumbs up**: thumb tip is above the wrist, all other fingers are folded
- **Thumbs down**: thumb tip is below the wrist, all other fingers are folded

In [ ]:
def detect_gesture(hand_landmarks):
    """
    Detect thumbs-up, thumbs-down, open-palm, fist, and pointing.
    Returns gesture name as a string.
    """
    lm = hand_landmarks.landmark

    # Wrist position
    wrist_y = lm[0].y
    thumb_tip_y = lm[4].y

    # Check if non-thumb fingers are folded (tip below pip)
    fingers_folded = all(
        lm[tip].y > lm[pip].y
        for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:])
    )

    # Thumbs up / down
    if fingers_folded:
        if thumb_tip_y < wrist_y - 0.05:   # thumb well above wrist
            return 'Thumbs Up'
        elif thumb_tip_y > wrist_y + 0.05:  # thumb well below wrist
            return 'Thumbs Down'
        return 'Fist'

    # Count extended fingers
    fingers_up = sum(
        lm[tip].y < lm[pip].y
        for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:])
    )

    if fingers_up == 4:
        return 'Open Palm'
    if fingers_up == 1 and lm[8].y < lm[6].y:   # index up only
        return 'Pointing'
    if fingers_up == 2 and lm[8].y < lm[6].y and lm[12].y < lm[10].y:
        return 'Peace / Victory'
    return f'{fingers_up} fingers'

# Demonstrate by synthesising gesture images
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
gesture_descriptions = [
    ('Thumbs Up',   'Thumb tip above wrist\nAll fingers folded'),
    ('Open Palm',   'All 5 fingers extended\nFull hand visible'),
    ('Pointing',    'Index finger extended\nOther fingers folded'),
]
for ax, (name, desc) in zip(axes, gesture_descriptions):
    canvas = np.ones((200, 180, 3), dtype=np.uint8) * 245
    cv2.putText(canvas, name, (10, 40),
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (50, 50, 200), 2)
    for i, line in enumerate(desc.split('\n')):
        cv2.putText(canvas, line, (10, 80 + i*35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80, 80, 80), 1)
    ax.imshow(canvas); ax.set_title(name, fontweight='bold')
    ax.axis('off')
plt.suptitle('Gesture Recognition Logic', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('Gesture detection rules:')
print('  Thumbs Up   → fingers_folded AND thumb_tip.y < wrist.y - 0.05')
print('  Thumbs Down → fingers_folded AND thumb_tip.y > wrist.y + 0.05')
print('  Open Palm   → all 4 non-thumb fingers extended')
print('  Pointing    → only index finger extended')
print('  Fist        → all fingers folded (including thumb)')

## Real-time Pipeline: Combining Pose + Hands

A complete real-time pipeline would process each video frame. Here we demonstrate the architecture with a single image.

In [ ]:
def full_pipeline(image_path):
    """
    Run pose estimation + hand tracking on a single image.
    Returns annotated frame with all detections.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        img_bgr = np.ones((480, 640, 3), dtype=np.uint8) * 200
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_bgr.shape[:2]

    annotated = img_bgr.copy()

    # --- Pose ---
    with mp_pose.Pose(static_image_mode=True,
                      min_detection_confidence=0.5) as pose:
        pose_results = pose.process(img_rgb)
        if pose_results.pose_landmarks:
            mp_drawing.draw_landmarks(
                annotated,
                pose_results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style()
            )

    # --- Hands ---
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2,
                        min_detection_confidence=0.5) as hands:
        hand_results = hands.process(img_rgb)
        if hand_results.multi_hand_landmarks:
            for hand_lm, hand_info in zip(
                    hand_results.multi_hand_landmarks,
                    hand_results.multi_handedness):
                mp_drawing.draw_landmarks(
                    annotated, hand_lm, mp_hands.HAND_CONNECTIONS)
                gesture = detect_gesture(hand_lm)
                wx = int(hand_lm.landmark[0].x * w)
                wy = int(hand_lm.landmark[0].y * h)
                cv2.putText(annotated, gesture, (wx - 40, wy - 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                            (255, 100, 0), 2, cv2.LINE_AA)

    # Status overlay
    pose_detected = pose_results.pose_landmarks is not None
    hand_count    = len(hand_results.multi_hand_landmarks) if hand_results.multi_hand_landmarks else 0
    cv2.rectangle(annotated, (0, 0), (300, 60), (0, 0, 0), -1)
    cv2.putText(annotated, f'Pose: {"Yes" if pose_detected else "No"}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.putText(annotated, f'Hands: {hand_count}',
                (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,200,255), 2)

    return annotated

pipeline_result = full_pipeline('/tmp/cv_pose/person.jpg')
show(pipeline_result, 'Full Pipeline: Pose + Hands Detection', figsize=(10, 7))

## Summary

| Solution | Landmarks | Key use case |
|---|---|---|
| `mp.solutions.pose` | 33 body landmarks | Exercise tracking, dance, sports analysis |
| `mp.solutions.hands` | 21 per hand | Gesture control, sign language |
| `mp.solutions.face_mesh` | 468 facial points | Face filters, expression recognition |

All MediaPipe solutions follow the same API:
```python
with mp.solutions.SOLUTION(config...) as detector:
    results = detector.process(rgb_image)
    # results.LANDMARKS_ATTRIBUTE contains the detections
```

## Practice Exercises

**Exercise 1 — Elbow Angle Calculator:**  
Using pose landmarks, compute the elbow flexion angle at each arm. The angle is the angle at the elbow joint formed by the shoulder-elbow-wrist vectors. Use the dot product formula: `cos(θ) = (a·b)/(|a||b|)`. Display the angle on the annotated image.

**Exercise 2 — Extended Finger Counter:**  
Capture 5 hand images (or use your webcam in Colab with `cv2.VideoCapture(0)`) showing different numbers of extended fingers (0–5). Run the finger counter on each and verify accuracy. Handle the left-hand / right-hand thumb direction correctly.

**Exercise 3 — Face Mesh Distance:**  
Using face mesh landmarks, compute the eye aspect ratio (EAR) to detect blink. EAR = (vertical eye distance) / (horizontal eye distance). Values below ~0.2 indicate a closed eye. Annotate the image with the EAR value for each eye.